# MNIST class 1 → 7: 32D pretrained VAE + 20-component PCA (no training)

**Goal:** Encode **all training images labeled 1 and 7** with a **pretrained 32D MNIST VAE**, compute two separate **20-component PCA bases**, choose the pair of eigenvectors with the largest *absolute* dot product, then test bounded scalar steps along the chosen **class-1** vector until the pretrained classifier predicts **7**. Visualize the original digit, VAE reconstruction, attacked decoder output, and image difference.

**No model training is performed in this notebook.** Pretrained weights are required for both models. The pretrained classifier downloads automatically; **you must supply a genuine pretrained 32D VAE whose model code and checkpoint match.** A checkpoint trained with latent size 16 cannot be used as size 32 by changing a variable.

### Mathematics

- `Z1`: `[N1, 32]`, `Z7`: `[N7, 32]`, using all MNIST **training** images labeled 1 or 7.
- `D1`, `D7`: `[32, 20]` orthonormal columns, each estimated from its own class.
- `S = D1.T @ D7`: `[20, 20]` with `S[i,j] = q_i.T @ r_j`.
- Choose `(i,j)` maximizing `abs(S[i,j])` because PCA axes have arbitrary signs.
- Let `q = D1[:, i]`. Test `z(alpha) = z0 + alpha*q` under `|alpha| <= EPSILON`.
- The scalar L2 norm is `||alpha||_2 = |alpha|`; because `||q||_2 = 1`, `||alpha*q||_2 = |alpha|`.

**Caveat:** parallel PCA axes do **not** prove movement toward class 7. A prediction of 7 does **not** prove the output still resembles a 1. Inspect the image manually.

In [ ]:
%pip -q install huggingface_hub

import os, sys, json, importlib, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM = 32
N_PCA = 20
EPSILON = 3.0      # example max |alpha| in latent units; adjust after inspection
ALPHA_STEPS = 150  # each magnitude tested in + and - directions
BATCH_SIZE = 256
OUT = Path('/content/pca_1to7_results')
OUT.mkdir(exist_ok=True, parents=True)
print('Device:', DEVICE)

## 1. Supply your pretrained **32D VAE**

This cell is intentionally configurable because different GitHub repositories define their encoder and decoder differently. You can either (A) clone your **actual pretrained 32D VAE** GitHub repo in the cell below, or (B) upload its Python model definition and matching checkpoint to Colab. **Do not use the old 16D checkpoint.**

Required methods after loading:

- `vae.encode_mu(images)` takes `[B,1,28,28]` and returns `[B,32]` (posterior means, deterministic).
- `vae.decode(latents)` takes `[B,32]` and returns `[B,1,28,28]` with values in `[0,1]`.

If your repository uses different method names, adapt the small `VAEAdapter` below. No retraining is involved.

In [ ]:
# Fill these values using the SAME repository that produced your pretrained 32D checkpoint.
# OPTION A (GitHub): enter a repo URL plus module/class/checkpoint details.
REPO_URL = ""  # e.g. https://github.com/OWNER/REPO.git
REPO_FOLDER = "/content/pretrained_vae32"
MODEL_MODULE = "model"   # e.g. "models.vae" or "vae"
MODEL_CLASS = "VAE"      # model constructor name from the repo
MODEL_KWARGS = {"latent_dim": 32}  # adjust according to its constructor
CHECKPOINT_PATH = ""    # e.g. /content/pretrained_vae32/checkpoints/vae32.pth

# OPTION B (upload): put your repo's Python model file and checkpoint in /content,
# then set MODEL_MODULE and CHECKPOINT_PATH above. E.g. model.py -> MODEL_MODULE='model'

if REPO_URL:
    if not Path(REPO_FOLDER).exists():
        subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_FOLDER], check=True)
    sys.path.insert(0, REPO_FOLDER)
else:
    print('No repo URL configured. Optionally upload model definition and checkpoint:')
    print('  from google.colab import files; files.upload()')
    print('Then set MODEL_MODULE and CHECKPOINT_PATH, and re-run this cell.')

In [ ]:
class VAEAdapter(nn.Module):
    """Adapt your pretrained VAE to fixed encode_mu/decode APIs."""
    def __init__(self, core):
        super().__init__()
        self.core = core

    def encode_mu(self, images):
        # Common VAE pattern: encode(images) -> (mu, logvar)
        if hasattr(self.core, 'encode_mu'):
            z = self.core.encode_mu(images)
        elif hasattr(self.core, 'encode'):
            z = self.core.encode(images)
            if isinstance(z, (tuple, list)):
                z = z[0]
            elif hasattr(z, 'loc'):
                z = z.loc
        else:
            raise TypeError('Adapt encode_mu: pretrained VAE has no compatible encoder method')
        if z.ndim > 2:
            z = z.flatten(start_dim=1)
        if z.ndim != 2 or z.shape[1] != 32:
            raise ValueError(f'Expected encoder output [batch,32], got {tuple(z.shape)}')
        return z

    def decode(self, latents):
        if not hasattr(self.core, 'decode'):
            raise TypeError('Adapt decode: pretrained VAE has no decode method')
        x = self.core.decode(latents)
        if isinstance(x, (tuple, list)):
            x = x[0]
        if x.ndim == 2 and x.shape[1] == 784:
            x = x.reshape(-1, 1, 28, 28)
        if x.shape != (latents.shape[0], 1, 28, 28):
            raise ValueError(f'Expected decoder [batch,1,28,28], got {tuple(x.shape)}')
        # IMPORTANT: do NOT blindly apply sigmoid if your decoder already outputs probabilities.
        if (x.min() < 0 or x.max() > 1):
            raise ValueError('Decoder output outside [0,1]. Adapt VAEAdapter.decode to apply the CORRECT output activation.')
        return x


def load_vae32():
    if not CHECKPOINT_PATH or not Path(CHECKPOINT_PATH).exists():
        raise FileNotFoundError(
            'Pretrained 32D VAE checkpoint not configured. Set CHECKPOINT_PATH '
            'and MODEL_MODULE/MODEL_CLASS/MODEL_KWARGS above; do not use a 16D checkpoint.'
        )
    module = importlib.import_module(MODEL_MODULE)
    cls = getattr(module, MODEL_CLASS)
    core = cls(**MODEL_KWARGS)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
    if isinstance(checkpoint, dict):
        for key in ['state_dict', 'model_state_dict', 'vae_state_dict']:
            if key in checkpoint:
                checkpoint = checkpoint[key]
                break
    if not isinstance(checkpoint, dict):
        raise ValueError('Checkpoint is not a weight state_dict. Adapt the loader to your repo format.')
    # Some training wrappers add a global prefix to every state key.
    raw_state = core.state_dict()
    if not set(raw_state).intersection(checkpoint):
        prefixes = ['module.', 'model.', 'vae.']
        for prefix in prefixes:
            stripped = {k[len(prefix):] if k.startswith(prefix) else k: v for k,v in checkpoint.items()}
            if set(raw_state).intersection(stripped):
                checkpoint = stripped
                break
    core.load_state_dict(checkpoint, strict=True)
    model = VAEAdapter(core).to(DEVICE).eval()
    with torch.no_grad():
        x = torch.rand(2, 1, 28, 28, device=DEVICE)
        mu = model.encode_mu(x)
        recon = model.decode(mu)
    print('Pretrained VAE validated:', tuple(mu.shape), '->', tuple(recon.shape))
    return model

# Uncomment and run AFTER configuring your 32D checkpoint:
# vae = load_vae32()

## 2. Download and load a **pretrained** MNIST classifier

This uses the publicly hosted **MNIST MLP** weights from [Vecrist/mnist-pytorch-digit-classifiers](https://huggingface.co/Vecrist/mnist-pytorch-digit-classifiers). The model card describes a 784→128→64→10 MLP. To avoid making assumptions about key prefixes, the loader verifies the weight shapes. It also compares common preprocessing choices using held-out MNIST images and warns if accuracy appears wrong. No classifier training occurs.

In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, image):
        return self.net(image)


def load_pretrained_classifier():
    ckpt = hf_hub_download(
        repo_id='Vecrist/mnist-pytorch-digit-classifiers',
        filename='MNIST_MLPmodel_weights.pth'
    )
    state = torch.load(ckpt, map_location='cpu', weights_only=True)
    if isinstance(state, dict):
        state = state.get('state_dict', state.get('model_state_dict', state))
    if not isinstance(state, dict):
        raise TypeError('Unexpected pretrained classifier checkpoint structure')

    model = MNISTMLP()
    expected = [(128, 784), (64, 128), (10, 64)]
    layers = [model.net[1], model.net[3], model.net[5]]
    for shape, layer in zip(expected, layers):
        candidates = [(k, v) for k,v in state.items()
                      if torch.is_tensor(v) and v.ndim == 2 and tuple(v.shape) == shape]
        if len(candidates) != 1:
            raise ValueError(f'Could not uniquely identify MLP weight shape {shape}: {len(candidates)} candidates')
        key, weight = candidates[0]
        bias_key = key[:-len('weight')] + 'bias'
        if bias_key not in state:
            raise ValueError(f'Missing bias for {key}')
        with torch.no_grad():
            layer.weight.copy_(weight)
            layer.bias.copy_(state[bias_key])

    model.to(DEVICE).eval()
    return model

@torch.no_grad()
def predict_batch(clf, images, normalization='raw'):
    if normalization == 'mnist_standard':
        images = (images - 0.1307) / 0.3081
    logits = clf(images)
    probabilities = torch.softmax(logits, dim=-1)
    return probabilities.argmax(dim=-1), probabilities

classifier = load_pretrained_classifier()
print('Classifier pretrained weights loaded')

## 3. Load MNIST and check classifier preprocessing

In [ ]:
train_set = datasets.MNIST('/content/mnist_data', train=True, download=True,
                           transform=transforms.ToTensor())
test_set = datasets.MNIST('/content/mnist_data', train=False, download=True,
                          transform=transforms.ToTensor())

check_loader = DataLoader(test_set, batch_size=256, shuffle=False)
imgs, labels = next(iter(check_loader))
imgs = imgs.to(DEVICE)

accuracy = {}
for mode in ['raw', 'mnist_standard']:
    y_pred, _ = predict_batch(classifier, imgs, normalization=mode)
    accuracy[mode] = (y_pred.cpu() == labels).float().mean().item()

NORMALIZATION = max(accuracy, key=accuracy.get)
print('Sanity-check accuracy on first 256 test images:', accuracy)
print('Chosen classifier input normalization:', NORMALIZATION)
if accuracy[NORMALIZATION] < 0.85:
    warnings.warn('Classifier performs poorly. Check model compatibility and preprocessing before trusting results.')

## 4. Encode **all** MNIST training images of classes 1 and 7

Encoder means are used for deterministic PCA. Running PCA separately on **each class** gives two eigenvector bases in the same 32D VAE latent space.

In [ ]:
vae = load_vae32()  # Will clearly stop here until a real 32D VAE is configured

@torch.no_grad()
def get_class_latents(dataset, vae, batch_size=BATCH_SIZE):
    # MNIST targets are directly accessible: select every 1 and 7 from training split.
    indices = torch.where((dataset.targets == 1) | (dataset.targets == 7))[0].tolist()
    loader = DataLoader(torch.utils.data.Subset(dataset, indices),
                        batch_size=batch_size, shuffle=False)
    z_by_class = {1: [], 7: []}
    for images, labels in loader:
        means = vae.encode_mu(images.to(DEVICE)).cpu()
        for cls in [1,7]:
            subset = means[labels == cls]
            if len(subset): z_by_class[cls].append(subset)
    return torch.cat(z_by_class[1]), torch.cat(z_by_class[7])

Z1, Z7 = get_class_latents(train_set, vae)
print('Class 1:', tuple(Z1.shape), 'Class 7:', tuple(Z7.shape))
assert Z1.shape[1] == Z7.shape[1] == 32

## 5. PCA → `D1` and `D7`, each **32 × 20**

In [ ]:
def fit_pca(Z, k=20):
    Z = Z.double()
    mean = Z.mean(dim=0)
    centered = Z - mean
    covariance = centered.T @ centered / Z.shape[0]
    evals, evecs = torch.linalg.eigh(covariance)
    rank_order = torch.argsort(evals, descending=True)
    select = rank_order[:k]
    D = evecs[:, select].float()
    top_evals = evals[select].float()
    assert D.shape == (32,20)
    assert torch.allclose(D.T @ D, torch.eye(k), atol=1e-4)
    return D, top_evals, mean.float(), covariance.float()

D1, eig1, mu1, cov1 = fit_pca(Z1, N_PCA)
D7, eig7, mu7, cov7 = fit_pca(Z7, N_PCA)
print('D1:',tuple(D1.shape),'D7:',tuple(D7.shape))
print('First five class-1 eigenvalues:',eig1[:5].tolist())
print('First five class-7 eigenvalues:',eig7[:5].tolist())

## 6. Compare all 20×20 eigenvector pairs

`S[i,j] = q_i.T @ r_j`. We use `abs(S)` only to rank the similarity of **unoriented PCA axes**. We later search both signs of the chosen class-1 direction.

In [ ]:
S = D1.T @ D7
absS = S.abs()
linear_index = int(absS.argmax())
i, j = np.unravel_index(linear_index, absS.shape)
i,j = int(i), int(j)
q = D1[:,i].to(DEVICE)  # class-1 PCA axis chosen by maximum pairwise alignment
r = D7[:,j].to(DEVICE)
print('S shape:',tuple(S.shape))
print(f'Pair: q_{i+1} and r_{j+1} (1-based numbering)')
print('Signed dot product:',float(S[i,j]))
print('Absolute alignment:',float(absS[i,j]))

plt.figure(figsize=(6,5))
plt.imshow(absS.numpy(), vmin=0, vmax=1, cmap='viridis')
plt.colorbar(label='|q_i^T r_j|')
plt.scatter([j],[i],marker='x',c='red',s=100)
plt.xlabel('Class-7 component index (0 based)')
plt.ylabel('Class-1 component index (0 based)')
plt.title('PCA eigenvector alignment')
plt.tight_layout()
plt.savefig(OUT/'alignment_heatmap.png',dpi=180)
plt.show()

# Save once: no need to repeat PCA on future runs using the same VAE
pca_payload={'D1':D1,'D7':D7,'S':S,'eig1':eig1,'eig7':eig7,
             'mu1':mu1,'mu7':mu7,'selected_i':i,'selected_j':j}
torch.save(pca_payload, OUT/'pca_1_7_32d_20pc.pt')

## 7. Choose a test image of digit 1 that the **decoded baseline** classifier calls 1

In [ ]:
@torch.no_grad()
def find_start_digit_one(dataset, vae, classifier, normalization):
    indices = torch.where(dataset.targets == 1)[0].tolist()
    for index in indices:
        original, _ = dataset[index]
        original = original.unsqueeze(0).to(DEVICE)
        z0 = vae.encode_mu(original)
        reconstruction = vae.decode(z0)
        predicted, probabilities = predict_batch(classifier, reconstruction, normalization)
        if int(predicted.item()) == 1:
            return original, z0, reconstruction, index
    raise RuntimeError('Could not find a test digit 1 whose VAE reconstruction is classified as 1')

original, z0, baseline, test_index = find_start_digit_one(test_set, vae, classifier, NORMALIZATION)
print('MNIST test index:',test_index,'latent shape:',tuple(z0.shape))
fig, axs=plt.subplots(1,2,figsize=(6,3))
axs[0].imshow(original[0,0].cpu(),cmap='gray',vmin=0,vmax=1)
axs[1].imshow(baseline[0,0].cpu(),cmap='gray',vmin=0,vmax=1)
for ax in axs:ax.axis('off')
axs[0].set_title('Original 1');axs[1].set_title('Unperturbed VAE reconstruction')
plt.show()

## 8. Bounded search over α until prediction 7 (no gradients needed)

We search increasing `|alpha|`, checking `+alpha` **and** `-alpha`. This is a discrete **grid search**, not a training/gradient step. If none succeeds within the budget, it reports that result and still shows the highest-P(7) candidate.

The constraint is `|alpha| <= EPSILON`; because `||q||=1`, the latent perturbation L2 norm is also `|alpha|`. Change `EPSILON` carefully, inspecting whether reconstructed images still resemble a 1.

In [ ]:
@torch.no_grad()
def bounded_alpha_search(z0, q, vae, classifier, norm_mode, epsilon, steps):
    q = q.reshape(1, 32)
    assert torch.allclose(torch.linalg.norm(q),torch.tensor(1.0,device=q.device),atol=1e-4)
    history=[]
    success=None
    best=None
    highest=-1.0
    step_magnitudes = np.linspace(0.0, epsilon, steps+1)

    for mag in step_magnitudes:
        candidates = [0.0] if mag == 0 else [float(mag),-float(mag)]
        for alpha in candidates:
            if abs(alpha) > epsilon + 1e-9: continue
            z_prime = z0 + alpha*q
            decoded = vae.decode(z_prime)
            pred, probs = predict_batch(classifier, decoded, norm_mode)
            p7 = float(probs[0,7])
            label = int(pred.item())
            history.append({'alpha':alpha,'L2_norm':abs(alpha),
                            'predicted_label':label,'probability_7':p7})
            snapshot = {'alpha':alpha,'predicted_label':label,'probability_7':p7,
                        'image':decoded.detach().cpu().clone()}
            if p7 > highest:
                best=snapshot
                highest=p7
            if label==7:
                success=snapshot
                break
        if success is not None:break
    return success, best, pd.DataFrame(history)

success,best,history = bounded_alpha_search(
    z0,q,vae,classifier,NORMALIZATION,EPSILON,ALPHA_STEPS
)
chosen = success if success is not None else best
print('SUCCESS' if success else 'No label-7 prediction in the tested alpha range')
print('Selected alpha:',chosen['alpha'],'L2:',abs(chosen['alpha']))
print('Selected prediction:',chosen['predicted_label'],'P(7):',chosen['probability_7'])
print('Number of model queries:',len(history))
history.to_csv(OUT/'alpha_history.csv',index=False)

## 9. Display original → reconstruction → attacked decoder output → change

In [ ]:
attacked = chosen['image'][0,0].numpy()
orig_np = original.detach().cpu()[0,0].numpy()
base_np = baseline.detach().cpu()[0,0].numpy()
diff = attacked - base_np

fig, axs = plt.subplots(1,4,figsize=(13,3.5))
for ax in axs:ax.axis('off')
axs[0].imshow(orig_np,cmap='gray',vmin=0,vmax=1)
axs[0].set_title('Original digit 1')
axs[1].imshow(base_np,cmap='gray',vmin=0,vmax=1)
axs[1].set_title('Decoder(z0)')
axs[2].imshow(attacked,cmap='gray',vmin=0,vmax=1)
axs[2].set_title(f"alpha={chosen['alpha']:+.3f} | pred={chosen['predicted_label']}")
scale=max(1e-6,float(np.max(np.abs(diff))))
axs[3].imshow(diff,cmap='seismic',vmin=-scale,vmax=scale)
axs[3].set_title('Decoder(z_prime) - Decoder(z0)')
plt.tight_layout();plt.savefig(OUT/'attack_result.png',dpi=200);plt.show()

history_sorted=history.sort_values('alpha')
plt.figure(figsize=(8,4))
plt.plot(history_sorted.alpha,history_sorted.probability_7,'o-',markersize=2)
plt.axvline(chosen['alpha'],linestyle='--',color='gray')
plt.xlabel('alpha (signed step)')
plt.ylabel('Classifier P(7)')
plt.title('Target-class response along selected class-1 PCA eigenvector')
plt.grid(alpha=.3);plt.tight_layout()
plt.savefig(OUT/'p7_by_alpha.png',dpi=180);plt.show()

print('Saved images, PCA matrices and search history in:', OUT)

## 10. Export results from Colab

The PCA basis is saved separately. On later runs with the **same pretrained VAE**, you may load `pca_1_7_32d_20pc.pt` instead of recomputing both PCAs.

In [ ]:
import shutil
archive=shutil.make_archive('/content/pca_1to7_results','zip',str(OUT))
print('Archive ready:',archive)
from google.colab import files
files.download(archive)